In [0]:
from pyspark.sql.functions import col, lit, expr, max
from delta.tables import DeltaTable


In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
df = spark.sql(f"""
               SELECT 
                EmployeeId,
                FirstName || ' ' || LastName AS FullName,
                StoreName
               FROM {catalog}.bronze.employees e
               INNER JOIN {catalog}.bronze.stores s
                ON e.StoreId = s.StoreId
               """)

In [0]:
df.createTempView("source_vw")

In [0]:
%sql
MERGE INTO IDENTIFIER(:catalog || ".silver.store_employees") tgt
USING source_vw src
ON tgt.EmployeeId = src.EmployeeId
WHEN MATCHED AND tgt.IsActive AND (tgt.FullName != src.FullName  OR tgt.StoreName != src.StoreName)
THEN UPDATE SET tgt.IsActive = FALSE, __end_at = current_date()
WHEN NOT MATCHED 
THEN INSERT (EmployeeId, FullName, StoreName, __start_at, __end_at, IsActive) VALUES (EmployeeId, FullName, StoreName, current_date(), NULL, 1)
WHEN NOT MATCHED BY SOURCE AND NOT tgt.IsActive
THEN DELETE;

INSERT INTO IDENTIFIER(:catalog || ".silver.store_employees")(EmployeeId, FullName, StoreName, __start_at, __end_at, IsActive)
SELECT s.EmployeeId, s.FullName, s.StoreName, current_date(), NULL, 1
FROM source_vw s
INNER JOIN IDENTIFIER(:catalog || ".silver.store_employees") t
  ON s.EmployeeId = t.EmployeeId  
WHERE t.IsActive = FALSE
AND t.__end_at = current_date();